In [10]:
#hide
import numpy as np
import plaquette
from plaquette.codes import LatticeCode
from plaquette import visualizer

from plaquette.errors import QubitErrorsDict
from plaquette.device import Device, MeasurementSample
from plaquette.circuit.generator import generate_qec_circuit

def get_syndrome_from_deterministic_error(code, qubit_errors: dict[int: list[str]], logical_ops="X"):
    qed: QubitErrorsDict = {
        "pauli": {i: {error: 1 for error in qubit_errors.get(i)} for i in qubit_errors.keys()}
    }
    circuit = generate_qec_circuit(code, qed, {}, logical_ops)
    device = Device("clifford")
    device.run(circuit)
    raw_results, erasure = device.get_sample()
    sample = MeasurementSample.from_code_and_raw_results(code, raw_results, erasure)
    return sample.syndrome[0]


def get_syndrome_from_random_error(code, qed, logical_ops="X"):
        circuit = generate_qec_circuit(code, qed, {}, logical_ops)
        device = Device("clifford")
        device.run(circuit)
        raw_results, erasure = device.get_sample()
        sample = MeasurementSample.from_code_and_raw_results(code, raw_results, erasure)
        return sample.syndrome[0]

For this code, the logical $\bar{Z}$ is defined by the application of $Z$ gates on the qubits lying at the bottom side of the lattice. The logical $\bar{X}$ is defined by the application of $X$ gates on the qubits lying on the left side of the lattice. The distance of a planar code is equal to the amount of data-qubits lying on a side of the lattice.

We can create a planar code using `plaquette`:

In [11]:
planar_code = LatticeCode.make_planar(size=3, n_rounds=1)

We use the `LatticeVisualizer` to inspect the code graphically. We use the same symbology for data-qubits, logical gates on the qubit, and entangling gates as in the previous representation. The only difference is that now we have two types of ancillas: $X$-type ancillas, represented by the blue tilted crosses, and $Z$-type ancillas, represented by the green crosses (in literature and in `plaquette` these are often reeffered to as A and B stabilizers).

In [12]:
vis = visualizer.LatticeVisualizer(planar_code)
vis.draw_lattice(height=300)


Let's take a look at how the syndrome looks given different errors on the code. You should try to place single qubit errors on any of the $d^2 + (d-1)^2$ data-qubits (where $d$ is the distance of the code) by making use of the previously-defined function `get_syndrome_from_deterministic_error`, to get a better idea of how every error affects the syndrome. By understanding this, you might start getting ideas of what errors could have produced more complicated codes and develop strategies to correct them (although there is no need to develop this strategies on your own, as we will see in the next chapter there already exist some ways of coming up with accurate corrections fast).


<div style="color: var(--nextui-colors-text); text-transform: none; background-color: white; margin: 20px 0px; padding: 20px; border-left: 3px solid;">
The planar code of distance $d=3$ with one logical qubit has $13$ data-qubits, so make sure that the indices used when you place the errors are not out of bounds!

-   `{i: ["x"]}` for any `i` between 0 and $d^2 + (d-1)^2 -1$.
-   `{i: ["z"]}` for any `i` between 0 and $d^2 + (d-1)^2 -1$.
-   `{i: ["y"]}` for any `i` between 0 and $d^2 + (d-1)^2 -1$.

</div>

In [13]:
syndrome = get_syndrome_from_deterministic_error(planar_code, {0: ["x"]})
vis.draw_latticedata(height=300, syndrome=syndrome)


You might have noticed that a $Y$ error acting on a qubit has the same effect on the syndrome as an $X$ error and a $Z$ error. This is because the Pauli operator $Y$ is proportional to the application of $XZ$ up to a global phase. This global phase does not show up in the syndrome or in any kind of measurements, because the outcome of a measurement is the eigenvalue of the measured state.

You could also try the following errors:

-   `{0: ["x"], 3: ["x"], 5: ["x"]}`
-   `{1: ["x"], 3: ["x"], 4: ["x"], 6: ["x"]}`
-   `{2: ["x"], 4: ["x"], 7: ["x"]}`
-   `{5: ["x"], 8: ["x"], 10: ["x"]}`
-   `{6: ["x"], 8: ["x"], 9: ["x"], 11: ["x"]}`
-   `{7: ["x"], 9: ["x"], 12: ["x"]}`
-   `{0: ["z"], 1: ["z"], 3: ["z"]}`
-   `{1: ["z"], 2: ["z"], 4: ["z"]}`
-   `{3: ["z"], 5: ["z"], 6: ["z"], 8: ["z"]}`
-   `{4: ["z"], 6: ["z"], 7: ["z"], 9: ["z"]}`
-   `{8: ["z"], 10: ["z"], 11: ["z"]}`
-   `{9: ["z"], 11: ["z"], 12: ["z"]}`

These errors, as you might have noticed, are the stabilizers themselves! Because they are the stabilizers, their action on the state leaves it unchanged (remember: $g_i \lvert\psi \rangle = \lvert\psi \rangle$). No stabilizers are toggled for these errors.

Can you guess what happens if you apply a logical error to the code?
These errors do *not* toggle any of the stabilizer measurements!

-   $\bar{X}$: `{0: ["x"], 1: ["x"], 2: ["x"]}`
-   $\bar{Z}$: `{0: ["z"], 5: ["z"], 20: ["z"]}`

More interesting error patterns can come out, and you are encouraged to test multiple errors of different types on several different qubits to explore how chains of errors might appear on the code. Here's a couple of suggestions:

-   `{3: ["x"], 9: ["z"]}`
-   `{3: ["z"], 9: ["x"]}`
-   `{0: ["x"], 6: ["z"], 12: ["y"]}`
-   `{3: ["x"], 4: ["x"]}`
-   `{3: ["z"], 4: ["z"]}`
-   `{3: ["y"], 4: ["y"]}`
-   `{3: ["x"], 6: ["x"]}`
-   `{3: ["z"], 6: ["z"]}`
-   `{3: ["y"], 6: ["y"]}`
-   `{3: ["x"], 4: ["x"], 8: ["x"]}`
-   `{3: ["z"], 4: ["z"], 8: ["z"]}`
-   `{3: ["x"], 6: ["x"], 9: ["x"]}`
-   `{3: ["z"], 6: ["z"], 9: ["z"]}`
-   `{3: ["y"], 4: ["y"], 8: ["y"]}`
-   `{3: ["y"], 6: ["y"], 9: ["y"]}`

These examples are instructive to see what happens to the syndrome in the presence of certain error patterns. We did cherry-pick some of these patterns to highlight particular effects (e.g. errors that do not toggle any syndrome bit), but it's time to abandon this "toy model". Errors do not appear deterministically as they have up until now, but rather they appear at random with a given probability. We can make use of `plaquette`'s simulator to model what happens when more realistic error models are taken into account. We might want to run a single scenario multiple times to see how errors randomly appear in the code and produce different syndromes.

We define the function `get_syndrome_from_random_error` to obtain a syndrome of a code with a specified error probability distribution. Given that now we are providing a probability distribution instead of deterministic instructions, multiple calls of this function with the same parameters might output different syndromes, because in one sample some errors might be present, and other errors might appear in a new sample. This function has three parameters:

-   `code`: an instance of `plaquette`'s error correction codes, `LatticeCode`.
-   `qed`: a dictionary of the type `plaquette.errors.QubitErrorData`, containing the error probability distribution.
-   `logical_ops`: a string necessary to create the quantum circuit used in simulations. At this moment, it is not necessary for you to worry about this last requirement.

In [14]:
def get_syndrome_from_random_error(code, qed, logical_ops="Z"):
    circuit = generate_qec_circuit(code, qed, {}, logical_ops)
    device = Device("clifford")
    device.run(circuit)
    raw_results, erasure = device.get_sample()
    sample = MeasurementSample.from_code_and_raw_results(code, raw_results, erasure)
    return sample.syndrome[0]

<div style="color: var(--nextui-colors-text); text-transform: none; background-color: white; margin: 20px 0px; padding: 20px; border-left: 3px solid;">
This function is very similar to `get_syndrome_from_deterministic_error`. The only change is that you must now provide an error distribution instead of deterministic instructions of where should errors be placed. The errors are chosen at random from the distribution and the syndrome for that sample is output.
</div>

Let's first begin by giving each of the $X$, $Y$ and $Z$ errors a probability of 0.2 of appearing on the data-qubit with index 6 (the one lying in the center of the planar code of distance 3). Feel free to run this cell as many times as you want to see that the outcome changes from time to time, depending on the error probabilities. There are even some cases where no error is applied.

In [15]:
qed: QubitErrorsDict = {
        "pauli": {6: dict(x=0.2, y=0.2, z=0.2)}
    }
syndrome = get_syndrome_from_random_error(planar_code, qed)
vis.draw_latticedata(height=300, syndrome=syndrome)

Now, let's try to see what would happen if we give every qubit in the code a probability of 0.1 of suffering an $X$, $Y$ and $Z$ errors. You might want to try to guess what errors acted on which qubits in order to give this syndrome. If you are unsure whether it was one error combination or another, remember that there are no wrong answers: in that case the syndrome is degenerate.

In [16]:
qed: QubitErrorsDict = {
        "pauli": {i: dict(x=0.1, y=0.1, z=0.1) for i in range(planar_code.n_data_qubits)}
    }
syndrome = get_syndrome_from_random_error(planar_code, qed)
vis.draw_latticedata(height=300, syndrome=syndrome)